# Transposable-element data

TE34 is the accession-linked genomic-resource panel. The LTR terminal:internal statistic is a mapping/deletion-footprint proxy, not a direct ectopic-recombination or DNA-loss rate.

This notebook reads only the compact canonical tables in this analysis directory. Change or add cells freely; the paper figures are built separately in R/ggplot.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'pyproject.toml').exists()
)
pd.set_option('display.max_columns', 100)

In [ ]:
import json

DATA = ROOT / 'analyses/01_transposable_elements/data'
panel = pd.read_csv(ROOT / 'data/identity/te34_panel.csv')
diversity = pd.read_csv(DATA / 'te_diversity.csv')
prevalence = pd.read_csv(DATA / 'te_feature_prevalence.csv')
clr = pd.read_csv(DATA / 'te_pca_clr_matrix.csv')
pca = pd.read_csv(DATA / 'te_pca_scores.csv')
variance = pd.read_csv(DATA / 'te_pca_variance.csv')
loadings = pd.read_csv(DATA / 'te_pca_loadings.csv')
diagnostics = pd.read_csv(DATA / 'te_pca_clustering_diagnostics.csv')
stability = pd.read_csv(DATA / 'te_pca_clustering_stability.csv')
candidates = pd.read_csv(DATA / 'te_pca_candidate_cluster_assignments.csv')
trait_tests = pd.read_csv(DATA / 'te_pca_trait_association_tests.csv')
phylogeny_tests = pd.read_csv(DATA / 'te_pca_phylogenetic_signal_tests.csv')
manifest = json.loads((DATA / 'te_pca_clustering_analysis_manifest.json').read_text())
landscape = pd.read_csv(DATA / 'repeatmasker_divergence_landscape.csv')
ltr = pd.read_csv(DATA / 'ltr_element_metrics.csv')

assert (panel['species'].nunique(), len(pca), len(variance), len(loadings)) == (34, 34, 23, 24)
assert manifest['selected_k'] is None
assert set(candidates['candidate_status']) == {'rejected'}
{'TE34 species': panel['species'].nunique(), 'PCA features': len(loadings), 'PCA axes': len(variance), 'landscape rows': len(landscape), 'usable LTR elements': len(ltr)}

In [ ]:
diversity.sort_values(['te_level', 'shannon_entropy'], ascending=[True, False]).head(20)

In [ ]:
display(
    pca[['species', 'PC1', 'PC2']].sort_values('PC1'),
    variance[['PC', 'variance_explained', 'cumulative_variance_explained']].head(10),
    loadings[['superfamily', 'PC1', 'PC2']].sort_values('PC1'),
)

In [ ]:
pd.Series({
    'analysis_id': manifest['analysis_id'],
    'analysis_status': manifest['analysis_status'],
    'species': manifest['pca_geometry']['species'],
    'ubiquitous_superfamilies': manifest['pca_geometry']['ubiquitous_superfamilies'],
    'PC1_PC2_variance': manifest['pca_geometry']['pc1_pc2_variance'],
    'gap_selected_k': manifest['clustering']['gap_primary_rule']['selected_k'],
    'accepted_k': manifest['clustering']['accepted_k'],
    'conclusion': manifest['clustering']['conclusion'],
    'tree_provenance': manifest['phylogenetic_signal']['provenance_status'],
}, name='final PCA audit')

In [ ]:
display(
    diagnostics.loc[diagnostics['representation'].eq('all_23_pcs'), [
        'k', 'mean_silhouette', 'minimum_cluster_size', 'singleton_count',
        'cluster_sizes_ascending', 'gap_supported', 'passes_feature_stability',
        'passes_species_stability', 'passes_six_pc_sensitivity',
        'accepted_cluster_solution',
    ]],
    stability[['k', 'stability_type', 'perturbation', 'n_replicates',
        'ari_q10', 'ari_median', 'ari_q90', 'fraction_ari_at_least_0_80']],
)

In [ ]:
trait_tests[['test_id', 'analysis_role', 'predictor', 'n_species',
    'statistic_name', 'statistic', 'variance_explained_r2',
    'p_value', 'adjusted_p_value', 'interpretation_boundary']]

In [ ]:
phylogeny_tests[['test_id', 'analysis_role', 'representation',
    'statistic_name', 'statistic', 'p_value', 'branch_length_use',
    'interpretation_boundary']]